# レッスン11 - エージェント間 (A2A) プロトコル


## セットアップ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A プロトコルとは何ですか？

**Agent-to-Agent（A2A）プロトコル**は、AIエージェントが通信し、
互いを発見し、協力することを可能にするオープンな標準です — 異なるフレームワークで構築されたり、異なるサービスでホストされている
場合でも。

主な概念:

- **Discovery** – エージェントは自分の能力を説明する*エージェントカード*を公開し、これにより
  他のエージェント（またはオーケストレータ）がタスクに適した専門家を見つけやすくなります。
- **Message Passing** – エージェントは共通のプロトコルを通じて構造化されたメッセージを交換するので、ある
  エージェントからのリクエストが理解され、
  内部実装に関係なく別のエージェントによって実行されます。
- **Task Lifecycle** – A2Aは*submitted*、*working*、*completed*、および
  *failed*を定義しており、オーケストレータに委任されたタスクの進行状況を完全に可視化します。

このレッスンでは、A2Aスタイルの協調をシミュレートするために、3つの専門化された旅行エージェントを
各エージェントが専門性を提供し、結果を次に渡すワークフローに組み込みます。


## 特化した旅行エージェントの作成


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ワークフローによるマルチエージェント協働

3つのエージェントをA2Aのメッセージパッシングを反映する逐次的なワークフローに接続します:

1. **CurrencyExchangeAgent** receives the user request and produces currency guidance.
2. **ActivityPlannerAgent** receives the enriched context and adds activity recommendations.
3. **TravelManagerAgent** synthesizes both inputs into a final travel brief.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 本番環境におけるA2Aの理解

本番環境では、A2Aプロトコルは強力なサービス間シナリオを可能にします:

| 機能 | 説明 |
|---|---|
| **フレームワーク間の相互運用** | あるフレームワークで構築されたエージェントが、他の任意のA2A対応フレームワークで構築されたエージェントにタスクを委任でき、組織間の真の相互運用性を実現します。 |
| **サービス境界** | エージェントは別々のマイクロサービス、クラウドリージョン、あるいは異なる組織に存在していても、シームレスに協調して動作できます。 |
| **動的ディスカバリ** | オーケストレーターは実行時にAgent Cardレジストリを照会して、特定のサブタスクに最適なスペシャリストを見つけることができます。 |
| **ストリーミング & プッシュ通知** | A2Aはリアルタイムの進捗更新のためにServer-Sent Events (SSE)をサポートし、長時間実行されるタスク向けのプッシュ通知も提供します。 |

上で構築したワークフローは、このパターンの簡略化されたプロセス内バージョンです。本番環境では
展開では各エージェントがHTTPエンドポイントを公開し、Agent Cardを発行し、
A2A JSON-RPCプロトコルを介して通信します。


## 概要

このレッスンで学んだこと:

1. **A2Aプロトコルとは何か** — エージェント間のディスカバリ、メッセージング、
   およびタスク管理のためのオープン標準。
2. **特化したエージェントの作り方** — 通貨交換エージェント、アクティビティプランナーエージェント、
   そしてトラベルマネージャーオーケストレーター。
3. **エージェントをワークフローに接続する方法** — `WorkflowBuilder` を使ってエージェント間の逐次的な
   メッセージ受け渡しをモデル化する。
4. **A2Aが本番運用でどのように機能するか** — フレームワークやサービスを越えたコラボレーションを可能にする
   動的なディスカバリとストリーミング更新。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責事項：
この文書は AI 翻訳サービス「Co-op Translator」（https://github.com/Azure/co-op-translator）を用いて翻訳されています。正確性の担保に努めていますが、自動翻訳には誤りや不正確な表現が含まれる可能性があります。原文（原言語の文書）を正式な情報源とみなしてください。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じた誤解や解釈の相違について、当方は一切の責任を負いません。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
